# Proof of Concept Pilot — Deepfake Detection (Group 2)

**Project:** Can We Trust AI-Image Detectors on Ugandan Content?
**Member:** AW
**Scope:** Tiny end-to-end pilot (~200 images) to prove the pipeline works.
**Not the final study.** Demonstrates data → model → evaluation → explanation.
Full-scale collection happens after this pilot passes.

### RAM-safe design
- No Hugging Face streaming buffers
- Only 200 images (100 real + 100 AI)
- 128×128 images, batch 8, 5 epochs
- Run cells one at a time — never "Run all"

### Drive paths
- Root: `/content/drive/MyDrive/Deepfake Detection Project/datasets`
- Pilot data: `pilot_data/`
- Results: `reports/`

In [ ]:
!pip -q install torch torchvision datasets imagehash tqdm scikit-learn pandas pillow

In [ ]:
import os, sys, json, glob, hashlib, time, gc
from datetime import datetime
from collections import Counter
import numpy as np
import pandas as pd
from PIL import Image as PILImage
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix, classification_report
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
print('PyTorch:', torch.__version__)
print('Imports OK')

## 1. Mount Drive + Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

ROOT = '/content/drive/MyDrive/Deepfake Detection Project/datasets'
MEMBER = 'AW'

N_REAL = 100
N_AI = 100
IMG_SIZE = 128
BATCH_SIZE = 8
EPOCHS = 5
LR = 1e-3
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

for d in ['pilot_data/real', 'pilot_data/ai', 'metadata', 'audit_outputs', 'reports']:
    os.makedirs(os.path.join(ROOT, d), exist_ok=True)

print('Drive root:', ROOT)
print(f'Pilot: {N_REAL} real + {N_AI} AI = {N_REAL + N_AI} images, {IMG_SIZE}x{IMG_SIZE}, batch {BATCH_SIZE}, {EPOCHS} epochs')

## 2. Download a tiny balanced subset from OpenFake
Streams only enough images to fill our target, then stops. No big shuffle buffers.

In [ ]:
from datasets import load_dataset
from datasets import Image as HFImage
import io

def download_tiny_openfake(root, n_real, n_ai, img_size):
    ds = load_dataset('ComplexDataLab/OpenFake', 'core', split='train', streaming=True)
    ds = ds.cast_column('image', HFImage(decode=False))
    
    real_dir = os.path.join(root, 'pilot_data', 'real')
    ai_dir = os.path.join(root, 'pilot_data', 'ai')
    os.makedirs(real_dir, exist_ok=True)
    os.makedirs(ai_dir, exist_ok=True)
    
    count_real, count_ai = 0, 0
    saved = []
    
    for i, ex in enumerate(ds):
        label = str(ex.get('label', '')).lower()
        if label == 'real' and count_real < n_real:
            raw = ex['image']['bytes'] if isinstance(ex['image'], dict) else ex['image']
            im = PILImage.open(io.BytesIO(raw)).convert('RGB').resize((img_size, img_size))
            fname = f'real_{count_real:04d}.jpg'
            im.save(os.path.join(real_dir, fname), quality=95)
            saved.append((os.path.join(real_dir, fname), 0, 'real', 'openfake'))
            count_real += 1
            print(f'Real: {count_real}/{n_real}', end='\r')
        elif label == 'fake' and count_ai < n_ai:
            raw = ex['image']['bytes'] if isinstance(ex['image'], dict) else ex['image']
            im = PILImage.open(io.BytesIO(raw)).convert('RGB').resize((img_size, img_size))
            fname = f'ai_{count_ai:04d}.jpg'
            im.save(os.path.join(ai_dir, fname), quality=95)
            saved.append((os.path.join(ai_dir, fname), 1, 'ai', 'openfake'))
            count_ai += 1
            print(f'AI:   {count_ai}/{n_ai}', end='\r')
        if count_real >= n_real and count_ai >= n_ai:
            break
    
    print()
    print(f'Downloaded {count_real} real + {count_ai} AI = {len(saved)} images')
    return saved

saved = download_tiny_openfake(ROOT, N_REAL, N_AI, IMG_SIZE)

## 3. Load images into tensors

In [ ]:
to_tensor = transforms.ToTensor()
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

X_list, y_list, meta_list = [], [], []

for path, label, class_name, source in saved:
    im = PILImage.open(path).convert('RGB')
    t = normalize(to_tensor(im))
    X_list.append(t)
    y_list.append(float(label))
    meta_list.append({'path': path, 'label': class_name, 'source': source})
    im.close()

X = torch.stack(X_list)
y = torch.tensor(y_list, dtype=torch.float32)
del X_list, y_list
gc.collect()

print('Tensor shape:', X.shape, '| Label shape:', y.shape)
print('Label counts:', Counter(y.int().numpy().tolist()))

## 4. Train / test split

In [ ]:
X_train, X_test, y_train, y_test, meta_train, meta_test = train_test_split(X, y, meta_list, test_size=0.2, random_state=SEED, stratify=y.int().numpy())

train_ds = TensorDataset(X_train, y_train)
test_ds = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

print(f'train: {len(train_ds)} | test: {len(test_ds)}')
print(f'train real={int((y_train==0).sum())} ai={int((y_train==1).sum())}')
print(f'test  real={int((y_test==0).sum())} ai={int((y_test==1).sum())}')

## 5. TinyCNN (trained from scratch — Run A baseline)

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self, img_size=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(64, 1),
        )
    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x.squeeze(1)

model = TinyCNN(img_size=IMG_SIZE).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

print('Model parameters:', sum(p.numel() for p in model.parameters()))
print(model)

## 6. Train (5 epochs)

In [ ]:
history = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, n = 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        n += xb.size(0)
    train_loss = total_loss / max(n, 1)
    
    model.eval()
    with torch.no_grad():
        probs = []
        for xb, _ in train_loader:
            xb = xb.to(DEVICE)
            probs.append(torch.sigmoid(model(xb)).cpu().numpy())
        train_probs = np.concatenate(probs)
    train_preds = (train_probs >= 0.5).astype(int)
    train_acc = accuracy_score(y_train.int().numpy(), train_preds)
    
    print(f'Epoch {epoch:2d} | train_loss={train_loss:.4f} | train_acc={train_acc:.4f}')
    history.append({'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc})

gc.collect()
print('Training complete.')

## 7. Evaluate on test set

In [ ]:
model.eval()
y_true, y_prob = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(DEVICE)
        logits = model(xb)
        probs = torch.sigmoid(logits).cpu().numpy()
        y_prob.extend(probs.tolist())
        y_true.extend(yb.numpy().tolist())

y_true = np.array(y_true).astype(int)
y_prob = np.array(y_prob)
y_pred = (y_prob >= 0.5).astype(int)

acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
try:
    auc = roc_auc_score(y_true, y_prob)
except ValueError:
    auc = float('nan')
cm = confusion_matrix(y_true, y_pred)

print('=== TEST RESULTS (Pilot) ===')
print(f'Accuracy : {acc:.4f}')
print(f'Macro-F1 : {f1:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall   : {rec:.4f}')
print(f'ROC AUC  : {auc:.4f}')
print('Confusion matrix (rows=true, cols=pred; 0=real, 1=ai):')
print(cm)
print()
print(classification_report(y_true, y_pred, target_names=['real', 'ai'], zero_division=0))

metrics = {
    'accuracy': float(acc),
    'macro_f1': float(f1),
    'precision': float(prec),
    'recall': float(rec),
    'roc_auc': float(auc),
    'confusion_matrix': cm.tolist(),
    'n_test': len(y_true),
    'model': 'TinyCNN (from scratch)',
    'member': MEMBER,
    'timestamp': datetime.utcnow().isoformat() + 'Z',
}
os.makedirs(os.path.join(ROOT, 'reports'), exist_ok=True)
with open(os.path.join(ROOT, 'reports', 'pilot_metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'\nMetrics saved to reports/pilot_metrics.json')

## 8. Grad-CAM on test samples

In [ ]:
features = None
gradients = None

def forward_hook(module, input, output):
    global features
    features = output.detach()

def backward_hook(module, grad_input, grad_output):
    global gradients
    gradients = grad_output[0].detach()

handle_f = model.features[-4].register_forward_hook(forward_hook)
handle_b = model.features[-4].register_full_backward_hook(backward_hook)

def compute_cam(x, target_class=1):
    model.zero_grad()
    logits = model(x)
    score = logits[:, target_class] if logits.dim() > 1 else logits[target_class]
    score.backward()
    weights = gradients.mean(dim=(2, 3), keepdim=True)
    cam = (weights * features).sum(dim=1, keepdim=True)
    cam = F.relu(cam)
    cam = F.interpolate(cam, size=(x.shape[2], x.shape[3]), mode='bilinear', align_corners=False)
    cam = cam.squeeze().cpu().numpy()
    if cam.max() > cam.min():
        cam = (cam - cam.min()) / (cam.max() - cam.min())
    return cam

test_real_idx = [i for i, l in enumerate(y_true) if l == 0][:4]
test_ai_idx = [i for i, l in enumerate(y_true) if l == 1][:4]
sample_idx = test_real_idx + test_ai_idx

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for row, idx in enumerate(sample_idx):
    x_single = X_test[idx:idx+1].to(DEVICE)
    cam = compute_cam(x_single)
    true_lab = 'real' if y_true[idx] == 0 else 'ai'
    pred_prob = y_prob[idx]
    pred_lab = 'ai' if pred_prob >= 0.5 else 'real'
    correct = '✓' if true_lab == pred_lab else '✗'
    
    with PILImage.open(meta_test[idx]['path']) as im:
        img_arr = np.array(im.resize((IMG_SIZE, IMG_SIZE)))
    
    axes[row, 0].imshow(img_arr)
    axes[row, 0].set_title(f'{true_lab} → {pred_lab} {correct}\nP(AI)={pred_prob:.2f}', fontsize=8)
    axes[row, 0].axis('off')
    
    axes[row, 1].imshow(cam, cmap='jet')
    axes[row, 1].set_title('Grad-CAM', fontsize=8)
    axes[row, 1].axis('off')
    
    axes[row, 2].imshow(img_arr)
    axes[row, 2].imshow(cam, cmap='jet', alpha=0.5)
    axes[row, 2].set_title('Overlay', fontsize=8)
    axes[row, 2].axis('off')
    
    axes[row, 3].axis('off')

plt.suptitle('Grad-CAM on pilot test samples (illustrative, not proof)', fontsize=11)
plt.tight_layout()
cam_path = os.path.join(ROOT, 'reports', 'pilot_gradcam.png')
plt.savefig(cam_path, dpi=150, bbox_inches='tight')
plt.close(fig)

handle_f.remove()
handle_b.remove()
gc.collect()
print('Grad-CAM saved to', cam_path)

## 9. Save model + presentation summary

In [ ]:
model_path = os.path.join(ROOT, 'reports', 'pilot_tinycnn.pt')
torch.save(model.state_dict(), model_path)
print('Model saved to', model_path)

pred_rows = []
for i in range(len(y_true)):
    pred_rows.append({
        'image_path': meta_test[i]['path'],
        'true_label': 'real' if y_true[i] == 0 else 'ai',
        'p_ai': float(y_prob[i]),
        'pred_label': 'ai' if y_pred[i] == 1 else 'real',
        'correct': bool(y_pred[i] == y_true[i]),
    })
pred_df = pd.DataFrame(pred_rows)
pred_csv = os.path.join(ROOT, 'reports', 'pilot_predictions.csv')
pred_df.to_csv(pred_csv, index=False)
print('Predictions saved to', pred_csv)

print()
print('=' * 60)
print('PILOT PROOF OF CONCEPT — PRESENTATION SUMMARY')
print('=' * 60)
print(f'Member: {MEMBER}')
print(f'Data:   {N_REAL} real + {N_AI} AI images from OpenFake (public, CC BY-NC 4.0)')
print(f'Model:  TinyCNN trained from scratch, {EPOCHS} epochs, batch {BATCH_SIZE}')
print(f'Input:  {IMG_SIZE}x{IMG_SIZE} RGB')
print()
print(f'Test accuracy : {acc:.4f}')
print(f'Test macro-F1 : {f1:.4f}')
print(f'Test ROC AUC  : {auc:.4f}')
print()
print('Confusion matrix (true rows, pred cols; 0=real, 1=ai):')
print(cm)
print()
print('LIMITATIONS (be honest about these):')
print(' - Tiny pilot: 200 images only, not representative of all Uganda.')
print(' - Single generator/source (OpenFake core).')
print(' - No WhatsApp compression, no held-out generators yet.')
print(' - Grad-CAM is illustrative, not proof of authenticity.')
print()
print('NEXT STEPS (expansion plan):')
print(' - Add Community Forensics + OpenFake Reddit sources.')
print(' - Scale to 2k–4k images per member (6 members total).')
print(' - Add Grad-CAM faithfulness test vs random region.')
print(' - Add WhatsApp-resent copies as robustness stress test.')
print(' - Lock Kampala pilot set for final held-out test.')
print('=' * 60)

gc.collect()
print('\nPilot complete. You can present this now.')